Numba is the most efficient way to clear a GPU. Now, why would we want that? Because, in this pipeline we'll have two LLMs, the mini lm to create the embeddings, the retrieval model, and we're going to also have the generative model. So normally, all of that doesn't fit into the GPU RAM, which means that we need a way to take our tensors, move them there, take our LLM, move it to the GPU, run our staff, take them out, and clear the GPU, and then do it on the next step without any kind of trash in the middle. That is very difficult to do. PyTorch and TensorFlow do not do it very well, but numba does.

In [1]:
!pip install requests tqdm faiss-cpu transformers tensorflow sentence-transformers textblob gensim numba

In [2]:
import os
import requests
import zipfile
from pathlib import Path
from tqdm import tqdm

# Directory to store downloaded and extracted data
DATA_DIR = Path("./mimic_textbooks")

# Download and extract the dataset zip file
def download_and_extract_zip(url, extract_to=DATA_DIR):
    # Ensure the directory exists
    extract_to.mkdir(parents=True, exist_ok=True)

    # Download the zip file
    zip_path = extract_to / "textbooks.zip"
    print("Downloading dataset...")
    response = requests.get(url, stream=True)
    with open(zip_path, "wb") as file:
        for chunk in tqdm(response.iter_content(chunk_size=1024), unit='KB'):
            if chunk:
                file.write(chunk)

    # Extract the zip file
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_to)
    print("Dataset downloaded and extracted.")

# Download and extract textbooks
dataset_url = "https://www.dropbox.com/scl/fi/54p9kkx5n93bffyx08eba/textbooks.zip?rlkey=2y2c5x8y0uncnddichn9cmd7n&st=m290nmkk&dl=1"
download_and_extract_zip(dataset_url)


88121KB [00:03, 27671.25KB/s]


Extracting dataset...
Dataset downloaded and extracted.


In [3]:
import re
from gensim.utils import simple_preprocess
from textblob import TextBlob

# Load text files
def load_text_files(directory):
    texts = []
    for file_path in Path(directory).glob("F*.txt"): # only grabbing the textbooks that start with an F to make this a little faster, to consume a little less memory, and so it feeds to a GPU T4, okay? In production environments, just remove the F, to get a production worthy code
        with open(file_path, "r", encoding="utf-8") as file:
            texts.append(file.read())
    return texts

# Cleaning and preprocessing function
def clean_and_tokenize(text):
    # Basic regex cleaning
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    text = text.lower()  # Lowercase all text
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters

    # Tokenize with gensim
    tokens = simple_preprocess(text)
    return ' '.join(tokens)

# Spell correction
def correct_spelling(text):
    return str(TextBlob(text).correct())

# Chunk text into fixed-size chunks
def chunk_text(text, chunk_size=200):
    words = text.split()
    return [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

# Load, clean, correct, and chunk documents
documents = load_text_files(DATA_DIR / "textbooks/en")
cleaned_documents = [clean_and_tokenize(doc) for doc in documents]
# corrected_documents = [correct_spelling(doc) for doc in cleaned_documents]
chunked_documents = []
for doc in cleaned_documents:
    chunked_documents.extend(chunk_text(doc))

print(f"Total document chunks created: {len(chunked_documents)}")


Total document chunks created: 1086


In [4]:
import re
from gensim.utils import simple_preprocess

CHUNK_SIZE = 200  # words per chunk

def load_text_files(directory):
    """Load all .txt files from a directory, returning (filename, text) tuples."""
    files = []
    for file_path in sorted(Path(directory).glob("F*.txt")):
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            files.append((file_path.name, f.read()))
    return files

def clean_and_tokenize(text):
    """Basic cleaning: normalise whitespace, lowercase, remove special chars."""
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    tokens = simple_preprocess(text)
    return ' '.join(tokens)

def chunk_text(text, chunk_size=CHUNK_SIZE):
    """Split text into fixed-size word chunks."""
    words = text.split()
    return [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

# Load, clean, and chunk — also track metadata (source file + chunk index)
raw_files = load_text_files(DATA_DIR / "textbooks/en")
print(f"Loaded {len(raw_files)} text files.")

chunked_documents = []  # list of chunk strings
chunk_metadata = []     # list of {"source": filename, "chunk_index": i}

for filename, text in raw_files:
    cleaned = clean_and_tokenize(text)
    chunks = chunk_text(cleaned)
    for i, chunk in enumerate(chunks):
        chunked_documents.append(chunk)
        chunk_metadata.append({"source": filename, "chunk_index": i})

print(f"Total document chunks: {len(chunked_documents)}")

# NOTE: Spell correction (textblob) is intentionally skipped.
# It is extremely slow on large medical corpora and rarely helps
# semantic similarity models which handle minor noise well.

Loaded 2 text files.
Total document chunks: 1086


In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Load model — downloads ~90MB on first run, cached afterwards
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

# Encode all chunks in batches with a progress bar
# Output shape: (num_chunks, 384)
embeddings = model.encode(
    chunked_documents,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # L2-normalise — cosine sim becomes dot product
)

print(f"Embeddings shape: {embeddings.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384


/tmp/ipykernel_8440/548443532.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")


Embeddings shape: (1086, 384)


In [6]:
import faiss
import numpy as np

# Define the dimension of embeddings
dimension = 384  # Embedding size from MiniLM model
index = faiss.IndexFlatL2(dimension)

# Convert embeddings to NumPy array for FAISS
embedding_matrix = np.array([embedding.flatten() for embedding in embeddings]).astype('float32')

# Add embeddings to FAISS index
index.add(embedding_matrix)
print(f"Total embeddings indexed: {index.ntotal}")


Total embeddings indexed: 1086


In [7]:
import torch
import gc
from numba import cuda

del model
torch.cuda.empty_cache()  # Clear GPU memory from torch
gc.collect()
device = cuda.get_current_device() # Clear GPU memory from tf
device.reset()

## Retrival Method

In [8]:
import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModel
import torch

# Load the tokenizer and model for retrieval on CPU
retrieval_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
retrieval_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").cpu()

# Function to generate embeddings for a new query
def get_query_embedding(query):
    with torch.no_grad():
        inputs = retrieval_tokenizer(query, return_tensors="pt", padding=True, truncation=True)
        outputs = retrieval_model(**inputs)
        embedding = torch.mean(outputs.last_hidden_state, dim=1).cpu().numpy()
    return embedding

# Load FAISS index with existing embeddings
embedding_dim = 384
index = faiss.IndexFlatL2(embedding_dim)

# Function to retrieve relevant documents based on the query
def retrieve_documents(query, top_k=5):
    query_embedding = get_query_embedding(query).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    results = [chunked_documents[idx] for idx in indices[0]]
    return results

# Test retrieval component
sample_query = "What are the symptoms of heart failure?"
similar_documents = retrieve_documents(sample_query)
print("Retrieved documents:", similar_documents)


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retrieved documents: ['crisis vestibuloocular ref ex scan aureus ventricular septal defect ventricular tachycardia von willebrands disease von willebrand factor varicellazoster virus white blood cell world health organization hemoglobin plasma mean corpuscular hemoglobin platelet count prothrombin time reticulocyte count sedimentation rate erythrocyte westergren proteins total mgdl moll mm pgcell fmolcell mm seconds seconds of red cells male mmh mmh female mmh mmh mg', 'crisis vestibuloocular ref ex scan aureus ventricular septal defect ventricular tachycardia von willebrands disease von willebrand factor varicellazoster virus white blood cell world health organization hemoglobin plasma mean corpuscular hemoglobin platelet count prothrombin time reticulocyte count sedimentation rate erythrocyte westergren proteins total mgdl moll mm pgcell fmolcell mm seconds seconds of red cells male mmh mmh female mmh mmh mg', 'crisis vestibuloocular ref ex scan aureus ventricular septal defect ventr

## Generation Method

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generation_tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct", trust_remote_code=True)
generation_model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3.5-mini-instruct", trust_remote_code=True, device_map="cpu", dtype=torch.bfloat16)
# The .to(device) call is removed as device_map handles placement.

In [ ]:

# Function to generate a response using retrieved context
def generate_response(query, context, max_new_tokens=100):
    input_text = f"User query: {query}\n\nContext:\n{context}\n\nAnswer:"

    # Tokenize the input and move tensors to GPU
    inputs = generation_tokenizer(input_text, return_tensors="pt", padding=True, truncation=True).to("cuda")

    # Generate response using max_new_tokens to control output length
    with torch.no_grad():
        outputs = generation_model.generate(inputs["input_ids"], max_new_tokens=max_new_tokens, num_return_sequences=1)

    # Decode the generated response
    response_text = generation_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response_text

# Testing generation with retrieved documents as context
retrieved_text = " ".join(similar_documents)  # Concatenate retrieved documents as context
response = generate_response(sample_query, retrieved_text)
print("Generated response:", response)

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


Generated response: User query: What are the symptoms of heart failure?

Context:
artery pip bisphosphate pip bisphosphate po partial pressure of oxygen pv plasma volume venous pressure correlation coefficient right variable group registration ranking results system rankl receptor activator of nuclear factor ligand rr relative risk respiratory rate rv residual volume right ventricle right ventricular se standard error of the mean siadh syndrome of inappropriate secretion of antidiuretic hormone sv splenic vein stroke volume tca tricarboxylic acid cycle tricyclic antidepressant vasopressin receptors vd volume of distribution vdj variable diversity joining gene segments rearranged to form ig genes vh variable region heavy chain antibody vl variable region light chain antibody vpl ventral posterior nucleus lateral vpm ventral posterior nucleus medial vpn vancomycin polymyxin nystatin media ratio xr xlinked recessive xxxy normal complement of sex chromosomes for femalemale zdv zidovudine f